In [22]:
# imports

import os
from openai import OpenAI
import google.generativeai as genai
from dotenv import load_dotenv
import gradio as gr

In [23]:
# constants

MODEL_GPT_NANO  = 'gpt-5-nano'
MODEL_GPT_4O    = 'gpt-4o-mini'
MODEL_GEMINI    = 'gemini-2.0-flash'

MODEL_OPTIONS = [MODEL_GPT_NANO, MODEL_GPT_4O, MODEL_GEMINI]

In [24]:
# set up environment

load_dotenv()

openai_api_key = os.getenv('OPENAI_API_KEY')
gemini_api_key = os.getenv('GEMINI_API_KEY')

if not openai_api_key:
    print("❌ OpenAI API key not found")
else:
    print("✅ OpenAI API key found!")
    openai_client = OpenAI(api_key=openai_api_key)

if not gemini_api_key:
    print("❌ Gemini API key not found")
else:
    print("✅ Gemini API key found!")
    genai.configure(api_key=gemini_api_key)

✅ OpenAI API key found!
✅ Gemini API key found!


In [25]:
# system prompt
system_prompt = (
    "You are a helpful technical assistant. "
    "When given a technical or coding question, provide a clear, "
    "concise explanation suitable for a software developer."
)

In [26]:
# helper functions

def transcribe(audio_filepath):
    with open(audio_filepath, "rb") as f:
        transcript = openai_client.audio.transcriptions.create(
            model="whisper-1",
            file=f
        )
    return transcript.text


def add_user_message(question, history):
    return "", history + [{"role": "user", "content": question}]


def add_audio_message(audio_filepath, history):
    if not audio_filepath:
        return history
    question = transcribe(audio_filepath)
    return history + [{"role": "user", "content": question}]

In [27]:
# streaming response function

def get_response(model, question, history):
    history_messages = [{"role": h["role"], "content": h["content"]} for h in history]
    messages = [{"role": "system", "content": system_prompt}] + history_messages

    if model in [MODEL_GPT_NANO, MODEL_GPT_4O]:
        stream = openai_client.chat.completions.create(
            model=model,
            messages=messages,
            stream=True
        )
        response = ""
        for chunk in stream:
            delta = chunk.choices[0].delta.content or ""
            response += delta
            yield history + [{"role": "assistant", "content": response}]

    elif model == MODEL_GEMINI:
        gemini = genai.GenerativeModel(
            model_name=MODEL_GEMINI,
            system_instruction=system_prompt
        )
        # Build Gemini-compatible history (excludes last user message)
        gemini_history = []
        for msg in history_messages[:-1]:
            role = "user" if msg["role"] == "user" else "model"
            gemini_history.append({"role": role, "parts": [msg["content"]]})

        chat = gemini.start_chat(history=gemini_history)
        stream = chat.send_message(history[-1]["content"], stream=True)
        response = ""
        for chunk in stream:
            response += chunk.text
            yield history + [{"role": "assistant", "content": response}]

In [28]:
#Gradio UI

with gr.Blocks() as ui:
    gr.Markdown("## 🤖 Technical Question Answerer")

    with gr.Row():
        model_picker = gr.Dropdown(
            choices=MODEL_OPTIONS,
            value=MODEL_GPT_NANO,
            label="Choose a model"
        )

    welcome_message = [{
        "role": "assistant",
        "content": (
            "👋 Welcome! I'm your technical question answerer. "
            "Ask me anything about code, programming concepts, or software development — "
            "you can type your question or use the microphone. "
            "You can also switch between models using the dropdown above!"
        )
    }]
    chatbot = gr.Chatbot(value=welcome_message, height=500, type="messages")

    with gr.Row():
        with gr.Column():
            question_box = gr.Textbox(
                lines=2,
                placeholder="Type your technical question here...",
                label="Your Question"
            )
            submit_btn = gr.Button("Ask", variant="primary")
        with gr.Column():
            audio_input = gr.Audio(
                sources="microphone",
                type="filepath",
                label="Or speak your question"
            )
            audio_btn = gr.Button("Ask with Audio", variant="secondary")

    # Text submit flow
    submit_btn.click(
        fn=lambda: gr.update(interactive=False),
        outputs=submit_btn
    ).then(
        fn=add_user_message,
        inputs=[question_box, chatbot],
        outputs=[question_box, chatbot]
    ).then(
        fn=get_response,
        inputs=[model_picker, question_box, chatbot],
        outputs=chatbot
    ).then(
        fn=lambda: gr.update(interactive=True),
        outputs=submit_btn
    )

    # Audio submit flow
    audio_btn.click(
        fn=lambda: gr.update(interactive=False),
        outputs=audio_btn
    ).then(
        fn=add_audio_message,
        inputs=[audio_input, chatbot],
        outputs=chatbot
    ).then(
        fn=get_response,
        inputs=[model_picker, question_box, chatbot],
        outputs=chatbot
    ).then(
        fn=lambda: gr.update(interactive=True),
        outputs=audio_btn
    )

ui.launch(inbrowser=True)

* Running on local URL:  http://127.0.0.1:7863
* To create a public link, set `share=True` in `launch()`.
